# 08. Análisis temporal de hostilidad alrededor de los momentos electorales

Este notebook analiza el corpus formal **media-anchored** con el perfil balanceado v2. Compara la hostilidad predicha antes, durante e inmediatamente después de cada momento, y describe su persistencia durante las 72 horas posteriores.

El análisis es descriptivo: una variación temporal no demuestra que el evento haya causado la hostilidad observada.

## 1. Objetivo

Responder cuatro preguntas:

1. ¿Cuál es el nivel de hostilidad predicha durante las 24 horas previas?
2. ¿Cómo cambia durante cada acontecimiento?
3. ¿Qué ocurre en las primeras 6 horas, entre 6 y 24 horas y entre 24 y 72 horas después?
4. ¿La hostilidad disminuye, permanece o aumenta durante el periodo posterior?

## 2. Entradas y salidas

**Entradas**

- `config/events.yaml`
- `data/processed/x_media_anchored_interactions_corpus_formal_predictions_v2.csv`

**Salidas**

- Tablas en `reports/formal_temporal/`
- Figuras en `reports/formal_temporal/figures/`

Este notebook no llama a la API de X ni modifica archivos de recolección, limpieza o etiquetado manual.

In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd
from IPython.display import Image, display


def find_project_root(start):
    for candidate in [start, *start.parents]:
        if (candidate / "config").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("No se pudo localizar la raíz del proyecto HateCR")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.temporal_analysis import (
    build_temporal_coverage,
    event_window_overlaps,
    load_event_schedule,
    prepare_temporal_corpus,
    save_phase_change_figure,
    save_phase_heatmap,
    save_post_event_decay_figure,
    summarize_persistence,
    summarize_post_event_bins,
    summarize_temporal_phases,
    temporal_assignments_table,
)

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
CONFIG_DIR = PROJECT_ROOT / "config"
REPORTS_DIR = PROJECT_ROOT / "reports" / "formal_temporal"
FIGURES_DIR = REPORTS_DIR / "figures"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)

## 3. Parámetros analíticos

Las fases son mutuamente excluyentes:

- `pre_24h`: 24 horas antes del inicio.
- `during`: desde el inicio hasta el final configurado del evento.
- `post_0_6h`: primeras 6 horas posteriores.
- `post_6_24h`: de 6 a 24 horas posteriores.
- `post_24_72h`: de 24 a 72 horas posteriores.

Los intervalos con menos de 20 comentarios se conservan para auditoría, pero se marcan como inestables y no deben sostener conclusiones.

In [ ]:
CORPUS_PATH = Path(
    os.getenv(
        "TEMPORAL_CORPUS_PATH",
        str(DATA_PROCESSED / "x_media_anchored_interactions_corpus_formal_predictions_v2.csv"),
    )
)
EVENTS_PATH = CONFIG_DIR / "events.yaml"
HOSTILITY_PREDICTION_COLUMN = os.getenv(
    "HOSTILITY_PREDICTION_COLUMN",
    "ml_hostility_pred_v2_balanced",
)
PRE_EVENT_HOURS = float(os.getenv("PRE_EVENT_HOURS", "24"))
MAX_POST_EVENT_HOURS = int(os.getenv("MAX_POST_EVENT_HOURS", "72"))
POST_BIN_HOURS = int(os.getenv("POST_BIN_HOURS", "6"))
MIN_PHASE_COMMENTS = int(os.getenv("MIN_PHASE_COMMENTS", "20"))
MIN_BIN_COMMENTS = int(os.getenv("MIN_BIN_COMMENTS", "20"))
DISPLAY_TIMEZONE = os.getenv("DISPLAY_TIMEZONE", "America/Costa_Rica")

PARAMETERS = pd.DataFrame(
    [
        {"parameter": "corpus", "value": str(CORPUS_PATH)},
        {"parameter": "prediction_column", "value": HOSTILITY_PREDICTION_COLUMN},
        {"parameter": "pre_event_hours", "value": PRE_EVENT_HOURS},
        {"parameter": "max_post_event_hours", "value": MAX_POST_EVENT_HOURS},
        {"parameter": "post_bin_hours", "value": POST_BIN_HOURS},
        {"parameter": "min_phase_comments", "value": MIN_PHASE_COMMENTS},
        {"parameter": "min_bin_comments", "value": MIN_BIN_COMMENTS},
        {"parameter": "display_timezone", "value": DISPLAY_TIMEZONE},
    ]
)
display(PARAMETERS)

## 4. Carga y validación

Las fronteras de los eventos se leen desde `events.yaml` y se convierten de hora de Costa Rica a UTC. El `event_id` heredado del post madre determina a qué momento pertenece cada interacción.

In [ ]:
def safe_read_csv(path, name):
    path = Path(path)
    if not path.exists():
        print("[ADVERTENCIA] No existe {}: {}".format(name, path))
        return pd.DataFrame()
    frame = pd.read_csv(path, dtype={"tweet_id": "string"})
    print("[OK] {}: {:,} filas".format(name, len(frame)))
    return frame


corpus_df = safe_read_csv(CORPUS_PATH, "corpus formal con predicciones v2")
if corpus_df.empty:
    raise FileNotFoundError(
        "No hay corpus para el análisis temporal. Revise TEMPORAL_CORPUS_PATH."
    )
if HOSTILITY_PREDICTION_COLUMN not in corpus_df.columns:
    raise KeyError(
        "No existe la columna de predicción: {}".format(HOSTILITY_PREDICTION_COLUMN)
    )

schedule_df = load_event_schedule(EVENTS_PATH, active_only=True, formal_only=True)

validation_df = pd.DataFrame(
    [
        {"check": "rows", "value": len(corpus_df)},
        {"check": "unique_tweets", "value": corpus_df["tweet_id"].nunique()},
        {"check": "duplicate_tweet_ids", "value": int(corpus_df["tweet_id"].duplicated().sum())},
        {"check": "missing_created_at", "value": int(corpus_df["created_at"].isna().sum())},
        {"check": "missing_event_id", "value": int(corpus_df["event_id"].isna().sum())},
        {
            "check": "missing_prediction",
            "value": int(corpus_df[HOSTILITY_PREDICTION_COLUMN].isna().sum()),
        },
        {"check": "formal_events", "value": len(schedule_df)},
    ]
)
display(validation_df)
display(
    schedule_df[
        [
            "formal_order",
            "event_id",
            "event_name",
            "event_start_utc",
            "event_end_utc",
            "event_duration_hours",
        ]
    ]
)

## 5. Preparación temporal y cobertura

Se deduplica por `tweet_id` únicamente dentro de la copia analítica. También se auditan observaciones fuera del rango de 24 horas antes a 72 horas después.

In [ ]:
corpus_unique_df = corpus_df.drop_duplicates("tweet_id", keep="first").copy()
prepared_df = prepare_temporal_corpus(
    corpus=corpus_unique_df,
    schedule=schedule_df,
    prediction_column=HOSTILITY_PREDICTION_COLUMN,
    display_timezone=DISPLAY_TIMEZONE,
    pre_event_hours=PRE_EVENT_HOURS,
    max_post_event_hours=MAX_POST_EVENT_HOURS,
)

phase_summary_df = summarize_temporal_phases(
    prepared_df,
    min_phase_comments=MIN_PHASE_COMMENTS,
)
post_bins_df = summarize_post_event_bins(
    prepared_df,
    bin_hours=POST_BIN_HOURS,
    max_post_event_hours=MAX_POST_EVENT_HOURS,
    min_bin_comments=MIN_BIN_COMMENTS,
)
persistence_df = summarize_persistence(
    phase_summary_df,
    post_bins_df,
    min_phase_comments=MIN_PHASE_COMMENTS,
    min_bin_comments=MIN_BIN_COMMENTS,
)
coverage_df = build_temporal_coverage(prepared_df, phase_summary_df)
overlap_df = event_window_overlaps(schedule_df)
assignments_df = temporal_assignments_table(prepared_df)

outside_df = (
    prepared_df.loc[~prepared_df["within_primary_temporal_window"]]
    .groupby(["event_id", "outside_reason"], dropna=False, as_index=False)
    .agg(n_comments=("tweet_id", "nunique"))
)

display(coverage_df)
if outside_df.empty:
    print("[OK] Todas las observaciones están dentro del rango temporal principal.")
else:
    print("[ADVERTENCIA] Observaciones fuera del rango temporal principal:")
    display(outside_df)

if overlap_df.empty:
    print("[OK] No hay solapamientos entre ventanas de recolección configuradas.")
else:
    print("[ADVERTENCIA] Hay ventanas configuradas que se solapan:")
    display(overlap_df)

## 6. Hostilidad antes, durante y después

Cada celda muestra el porcentaje predicho como hostil y el número de comentarios que lo sustenta. El símbolo `†` identifica fases con muestra menor al mínimo configurado.

In [ ]:
phase_display_columns = [
    "formal_order",
    "event_name",
    "temporal_phase",
    "comment_count",
    "hostile_count",
    "hostility_pct",
    "hostility_ci95_low",
    "hostility_ci95_high",
    "comments_per_hour",
    "low_n_flag",
]
display(phase_summary_df[phase_display_columns].round(2))

PHASE_FIGURE = save_phase_heatmap(
    phase_summary_df,
    FIGURES_DIR / "hostility_before_during_after_heatmap.png",
    min_phase_comments=MIN_PHASE_COMMENTS,
)
display(Image(filename=str(PHASE_FIGURE)))

## 7. Persistencia y velocidad de cambio

La serie posterior divide las 72 horas en intervalos de 6 horas. La línea previa corresponde a las 24 horas anteriores al evento. Los intervalos con muestra insuficiente se dibujan como cruces y no se usan para calcular la pendiente.

La pendiente se expresa en **puntos porcentuales por 24 horas**. Resume una tendencia lineal descriptiva; no implica causalidad ni que la hostilidad individual se comporte linealmente.

In [ ]:
PERSISTENCE_FIGURE = save_post_event_decay_figure(
    post_bins_df,
    phase_summary_df,
    FIGURES_DIR / "hostility_post_event_decay_72h.png",
    min_bin_comments=MIN_BIN_COMMENTS,
    min_pre_comments=MIN_PHASE_COMMENTS,
)
display(Image(filename=str(PERSISTENCE_FIGURE)))

persistence_columns = [
    "formal_order",
    "event_name",
    "pre_24h_n",
    "pre_24h_hostility_pct",
    "during_n",
    "during_hostility_pct",
    "post_0_24h_n",
    "post_0_24h_hostility_pct",
    "post_24_72h_n",
    "post_24_72h_hostility_pct",
    "post_trend_change_per_24h_pp",
    "valid_post_bins",
    "temporal_reliability",
]
display(persistence_df[persistence_columns].round(2))

## 8. Cambios respecto al nivel previo

Los cambios se calculan en puntos porcentuales frente a las 24 horas anteriores. Las comparaciones se omiten si la cobertura temporal necesaria es insuficiente.

In [ ]:
CHANGE_FIGURE = save_phase_change_figure(
    persistence_df,
    FIGURES_DIR / "hostility_change_vs_pre_event.png",
)
display(Image(filename=str(CHANGE_FIGURE)))

## 9. Interpretación descriptiva automática

Las etiquetas cualitativas siguientes ayudan a leer las magnitudes, pero no sustituyen la interpretación histórica ni la revisión de los intervalos de confianza y tamaños muestrales.

In [ ]:
def describe_level_change(value):
    if pd.isna(value):
        return "sin_datos"
    if value >= 3:
        return "mayor_que_nivel_previo"
    if value <= -3:
        return "menor_que_nivel_previo"
    return "cercano_al_nivel_previo"


def describe_post_trend(value):
    if pd.isna(value):
        return "sin_tendencia_estimable"
    if value >= 1.5:
        return "aumento_postevento"
    if value <= -1.5:
        return "descenso_postevento"
    return "relativamente_estable"


interpretation_df = persistence_df.copy()
interpretation_df["during_vs_pre_interpretation"] = interpretation_df[
    "during_minus_pre_pp"
].map(describe_level_change)
interpretation_df["early_post_vs_pre_interpretation"] = interpretation_df[
    "post_0_24h_minus_pre_pp"
].map(describe_level_change)
interpretation_df["late_post_vs_pre_interpretation"] = interpretation_df[
    "post_24_72h_minus_pre_pp"
].map(describe_level_change)
interpretation_df["post_trajectory_interpretation"] = interpretation_df[
    "post_trend_change_per_24h_pp"
].map(describe_post_trend)

limited = interpretation_df["temporal_reliability"].astype(str).str.startswith("limited:")
interpretation_columns = [
    "formal_order",
    "event_id",
    "event_name",
    "during_minus_pre_pp",
    "post_0_24h_minus_pre_pp",
    "post_24_72h_minus_pre_pp",
    "post_trend_change_per_24h_pp",
    "during_vs_pre_interpretation",
    "early_post_vs_pre_interpretation",
    "late_post_vs_pre_interpretation",
    "post_trajectory_interpretation",
    "temporal_reliability",
]
interpretation_df.loc[
    limited,
    [
        "during_vs_pre_interpretation",
        "early_post_vs_pre_interpretation",
        "late_post_vs_pre_interpretation",
        "post_trajectory_interpretation",
    ],
] = "no_interpretar_cobertura_insuficiente"

display(interpretation_df[interpretation_columns].round(2))

## 10. Exportación

Se guardan tablas agregadas y una auditoría por comentario sin el texto. El corpus original permanece intacto.

In [ ]:
schedule_df.to_csv(REPORTS_DIR / "event_schedule_utc.csv", index=False)
phase_summary_df.to_csv(
    REPORTS_DIR / "hostility_temporal_phase_summary.csv", index=False
)
post_bins_df.to_csv(
    REPORTS_DIR / "hostility_post_event_6h_bins.csv", index=False
)
persistence_df.to_csv(
    REPORTS_DIR / "hostility_temporal_persistence_summary.csv", index=False
)
coverage_df.to_csv(
    REPORTS_DIR / "hostility_temporal_coverage.csv", index=False
)
overlap_df.to_csv(REPORTS_DIR / "event_window_overlaps.csv", index=False)
assignments_df.to_csv(
    REPORTS_DIR / "comment_temporal_assignments.csv", index=False
)
interpretation_df[interpretation_columns].to_csv(
    REPORTS_DIR / "hostility_temporal_interpretation.csv", index=False
)

output_files = sorted(
    [*REPORTS_DIR.glob("*.csv"), *FIGURES_DIR.glob("*.png")],
    key=lambda path: str(path),
)
print("Salidas generadas:")
for output_file in output_files:
    print(" -", output_file.relative_to(PROJECT_ROOT))

## 11. Advertencias metodológicas

- Las etiquetas son **predicciones del perfil balanceado v2**, no anotaciones humanas de todo el corpus ni estimaciones definitivas de prevalencia.
- El corpus está anclado en publicaciones de medios costarricenses; no representa toda la conversación política en X.
- Las diferencias son asociaciones temporales descriptivas. No permiten atribuir causalidad al evento.
- Los eventos tienen duraciones distintas. Por eso se muestran tanto proporciones como comentarios por hora.
- El evento de Repretel/Radio Monumental y el cierre de campaña tienen ventanas configuradas que se solapan. El `event_id` heredado del post madre mantiene las filas separadas, pero la cercanía histórica debe considerarse al interpretar resultados.
- Fases o intervalos con pocos comentarios producen porcentajes inestables. El notebook los marca y excluye de comparaciones agregadas cuando corresponde.
- La pendiente postevento resume intervalos válidos mediante una recta ponderada por volumen. No describe trayectorias individuales ni garantiza un decaimiento lineal.

La interpretación final debe combinar estas tablas con revisión cualitativa de comentarios, contexto político y documentación de las decisiones de muestreo.